In [1]:
import os

notebookPath = os.path.dirname(os.path.dirname(os.getcwd()))
output_dir = os.path.join(notebookPath, "data", "new")
pastas = sorted(os.listdir(output_dir))

In [2]:
import scanpy as sc
bdata = sc.read('/mnt/SATA/singleCellTaru/data/adata_harmony.h5ad')

In [10]:
bdata.layers["norm"] = bdata.X.copy()

In [11]:
bdata.X = bdata.layers["counts"].copy()

In [12]:

from decontx import decontx
# Rodar decontX
result = decontx(
    adata=bdata,
    batch_key='library_id',
    seed=42,
    verbose=True,
    cluster_key="leiden_0.5"
)

Starting DecontX
Processing 34969 cells, 5000 genes
Using 9 clusters from 'leiden_0.5'
Processing 14 batches separately
  Processing batch 'CTH92_ML'...
    Mean contamination: 26.8%
  Processing batch 'CTH92_NS'...
    Mean contamination: 20.5%
  Processing batch 'CTH94_ML'...
    Mean contamination: 19.9%
  Processing batch 'CTH94_NS'...


SystemError: CPUDispatcher(<function decontx_em_exact at 0x7f1429ac23e0>) returned a result with an exception set

In [ ]:
import os
import scanpy as sc
import numpy as np
import scipy.sparse as sp

# =============================================================
# 0. Diretório de saída
# =============================================================
output_dir = os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), "data", "filtered_h5ad")
os.makedirs(output_dir, exist_ok=True)

# =============================================================
# 1. Carregar amostras como LISTA
# =============================================================

adata_list = [sc.read_h5ad(os.path.join(output_dir, p)) for p in pastas]

# =============================================================
# 2. Rodar decontX por amostra (correto)
# =============================================================
adata_clean = []

for adata, nome in zip(adata_list, pastas):
    print(f"Processando: {nome}")

    # Garantir counts brutos
    if "counts" not in adata.layers:
        adata.layers["counts"] = adata.X.copy()

    # Rodar decontX
    result = decontx(
        adata=adata,
        batch_key=None,
        seed=42,
        verbose=True
    )

    # Adicionar resultados
    adata.obs["decontX_contamination"] = result["contamination"]
    adata.layers["decontXcounts"] = sp.csr_matrix(result["decontXcounts"])

    # Substituir matriz principal
    adata.X = adata.layers["decontXcounts"].copy()

    # =============================================================
    # 3. Filtrar células
    # =============================================================
    print(f"Células antes: {adata.n_obs}")
    adata = adata[adata.obs["decontX_contamination"] < 0.3].copy()
    print(f"Células depois: {adata.n_obs}")

    adata_clean.append(adata)

    sc.pl.umap(
        adata,
        color="decontX_contamination",
        cmap="viridis",
        vmax=0.5  # opcional: corta outliers
    )